# Fusion model (LiDAR + multispectral)

## Importing parameters

In [1]:
import os, glob, re
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.features import rasterize
from shapely.geometry import box

In [2]:
ROOT_LIDAR = r"D:/Kiran/Harz_Results" 
ROOT_MS = r"D:/Kiran/Harz_Results/QGIS_Harz" # Multispectral (MS)
PLOT_BUFF = r"D:/Kiran/Harz_Results/Plot_buff10/Plot_buff10.shp"
DW_DOWN = r"D:/Kiran/Harz_Results/QGIS_Harz/Digitizing_KI Recover/Downed Deadwood_EPSG32632.shp"
DW_STAND = r"D:/Kiran/Harz_Results/QGIS_Harz/Digitizing_KI Recover/Standing Deadwood.shp" 
OUT_DIR = r"D:/Kiran/Harz_Results/Modeling_results"

## Defining metric folders

In [3]:
LIDAR_FOLDERS = {
    "CHM": os.path.join(ROOT_LIDAR, "CHMs"),
     "H95": os.path.join(ROOT_LIDAR, "H95s"),
      "H10": os.path.join(ROOT_LIDAR, "H10s"),
       "CGF2m": os.path.join(ROOT_LIDAR, "GapFrac_below2m"),
        "CC2m": os.path.join(ROOT_LIDAR, "Cover_above2m"),
         "Hmean": os.path.join(ROOT_LIDAR, "Hmeans"),
          "Hmed": os.path.join(ROOT_LIDAR, "Hmeds"),
           "Hsd": os.path.join(ROOT_LIDAR, "Hsds"),
            "IQR": os.path.join(ROOT_LIDAR, "IQRs"),
} 

In [4]:
MS_FOLDERS = {
    "NDVI": os.path.join(ROOT_MS, "NDVIs"),
     "SAVI": os.path.join(ROOT_MS, "SAVIs"),
      "GNDVI": os.path.join(ROOT_MS, "GNDVIs"),
       "GCI": os.path.join(ROOT_MS, "CIgs"),
        "EVI": os.path.join(ROOT_MS, "EVIs"),
         "MSAVI": os.path.join(ROOT_MS, "MSAVIs"),
          "NDRE": os.path.join(ROOT_MS, "NDREs"),
           "VARI": os.path.join(ROOT_MS, "VARIs"),  
}

## Loading vectors

In [5]:
gdf_plot = gpd.read_file(PLOT_BUFF)
gdf_down = gpd.read_file(DW_DOWN)
gdf_stand = gpd.read_file(DW_STAND)

## Helper: Listing sites from CHM rasters

In [6]:
def list_sites_from_chm(chm_folder):
   tifs = glob.glob(os.path.join(chm_folder, "*.tif")) + glob.glob(os.path.join(chm_folder, "*.tiff"))
   sites = []
   for p in tifs:
      base = os.path.splitext(os.path.basename(p))[0]
      m = re.search(r"CHM_([A-Za-z]+[0-9]+)", base)
      if m:  
        sites.append(m.group(1).upper())
   return sorted(list(set(sites)))

SITES = list_sites_from_chm(LIDAR_FOLDERS["CHM"])
print("Found sites:", SITES)

Found sites: ['CHH1', 'CHH2', 'CHH3', 'CHH4', 'GB1', 'GB2', 'GB3', 'GOS1', 'GOS2', 'MGH1', 'MGH2', 'MGH3', 'MGH4', 'MGH5', 'MGH6', 'MGH7', 'MK1', 'MK2', 'MK3', 'MK4', 'QB1', 'QB2', 'QB3', 'QB4', 'QB5', 'WW1', 'WW2']


## Helper: Finding the raster for a given metric + site even with different naming

In [7]:
def find_raster_for_site(folder, site_id, must_contain=None):
     site_id = site_id.upper()
     code = re.match(r"([A-Z]+)", site_id).group(1)
     num = re.match(r"[A-Z]+([0-9]+)", site_id).group(1)
    
     candidates = glob.glob(os.path.join(folder, "*.tif")) + glob.glob(os.path.join(folder, "*.tiff"))
     scored = []

     for p in candidates:
         name = os.path.splitext(os.path.basename(p))[0].upper()
         s = 0
         if code in name: s += 10
         if site_id in name: s += 20
         if re.search(rf"{code}\D*{num}\b", name): s += 15 
         if must_contain is not None and must_contain.upper() in name: s += 5
         if s > 0: scored.append((s, p))
     if not scored:
             return None

     scored.sort(key=lambda x: x[0], reverse=True)
     top_score = scored[0][0]
     top = [p for s, p in scored if s == top_score]
     return top[0]
    

## Helper: Resampling MS raster to LiDAR reference grid

In [8]:
def resample_to_ref(src_path, ref_ds):
    with rasterio.open(src_path) as src:
        dst = np.full((ref_ds.height, ref_ds.width), np.nan, dtype=np.float32)
        reproject(
            source=rasterio.band(src, 1),
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=ref_ds.transform,
            dst_crs=ref_ds.crs,
            resampling=Resampling.bilinear
        )
    return dst
        

## Helper: Making plot-buffer mask on the LiDAR grid 

In [9]:
def buffers_for_this_raster(gdf_buffers, ref_ds):
   xmin, ymin, xmax, ymax = ref_ds.bounds
   bbox = box(xmin, ymin, xmax, ymax)
   sel = gdf_buffers[gdf_buffers.intersects(bbox)].copy()
   return sel

def rasterize_mask(ref_ds, geoms, burn_value=1, fill_value=0, dtype=np.uint8):
    shapes = ((geom, burn_value) for geom in geoms)
    m = rasterize(
        shapes=shapes,
        out_shape=(ref_ds.height, ref_ds.width),
        transform=ref_ds.transform,
        fill=fill_value,
        dtype=dtype
    )
    return m

## Helper: Rasterizing standing + downed deadwood labels on LiDAR grid

In [10]:
def make_deadwood_label_raster(ref_ds, gdf_stand_site, gdf_down_site):
    y = np.full((ref_ds.height, ref_ds.width), 0, dtype=np.uint8)
    if len(gdf_stand_site) > 0:  # if standing deadwood exists #
        m_st = rasterize_mask(ref_ds, gdf_stand_site.geometry, burn_value=1, fill_value=0, dtype=np.uint8)  # class 1 mask #
        y[m_st == 1] = 1  # set class 1 = standing #
    if len(gdf_down_site) > 0:  # if downed deadwood exists #
        m_dn = rasterize_mask(ref_ds, gdf_down_site.geometry, burn_value=1, fill_value=0, dtype=np.uint8)  # class 2 mask #
        y[m_dn == 1] = 2
    return y  # 0=unlabeled, 1=standing, 2=downed #

## Build training data across ALL sites (LiDAR already clipped, MS gets clipped via plot mask)

In [11]:
FEATURE_NAMES = []  # will be filled once from metrics used #
X_list = []  # list of training feature rows #
y_list = []  # list of training labels #

BG_CLASS = 3  # class 3 = background #
RANDOM_SEED = 42  # reproducibility #
np.random.seed(RANDOM_SEED)  # set random seed #

for site_id in SITES:  # loop all 27 sites #
    chm_path = find_raster_for_site(LIDAR_FOLDERS["CHM"], site_id, must_contain="CHM")  # CHM reference raster #
    if chm_path is None:  # if missing #
        print("Missing CHM for", site_id, "-> skipping")  # report #
        continue  # skip site #

    with rasterio.open(chm_path) as ref:  # open CHM as the reference grid #
        # --- a) Select plot buffers that intersect this site's LiDAR raster --- #
        buff_site = buffers_for_this_raster(gdf_plot, ref)  # buffers that overlap raster extent #
        if len(buff_site) == 0:  # if no buffers overlap #
            print("No plot buffers intersect", site_id, "-> skipping")  # report #
            continue  # skip #

        plot_mask = rasterize_mask(ref, buff_site.geometry, burn_value=1, fill_value=0, dtype=np.uint8).astype(bool)  # True inside buffers #
        # NOTE: plot_mask is the exact "clip-to-plot-buffer" step (we only use pixels where plot_mask==True) #  
        
        # --- b) Filter deadwood vectors to this raster extent (spatial, not name-based) --- #
        buff_union = buff_site.geometry.union_all()
        stand_site = gdf_stand[gdf_stand.intersects(buff_union)].copy()  # standing deadwood near these plots #
        down_site = gdf_down[gdf_down.intersects(buff_union)].copy()  # downed deadwood near these plots #

        # --- b.1) Buffer standing deadwood points (10 cm) BEFORE rasterizing --- #
        # IMPORTANT: buffer distance is in the CRS units; ensure vectors are in the same projected CRS as the raster (meters) #
        if ref.crs is not None:  # align vector CRS to raster CRS if needed #
            if stand_site.crs != ref.crs:  
                stand_site = stand_site.to_crs(ref.crs)  # reproject to raster CRS #
            if down_site.crs != ref.crs: 
                down_site = down_site.to_crs(ref.crs)

        stand_site['geometry'] = stand_site.geometry.buffer(0.10)  # 10 cm buffer around each digitized standing deadwood point #

        # --- c) Rasterize standing/downed labels onto LiDAR grid --- #
        y_sd = make_deadwood_label_raster(ref, stand_site, down_site)  # 1=standing, 2=downed, 0=unlabeled #

        # --- d) Load LiDAR metrics (already clipped) into feature stack --- #
        lidar_arrays = []  # store LiDAR feature arrays #
        lidar_used = []  # store LiDAR metric names #
        for met, folder in LIDAR_FOLDERS.items():  # loop LiDAR metric folders #
            p = find_raster_for_site(folder, site_id, must_contain=met)  # find the site raster #
            if p is None: 
                print("Missing LiDAR", met, "for", site_id, "-> not used")
                continue 
            with rasterio.open(p) as src:  # open metric raster #
                a = src.read(1).astype(np.float32)
            lidar_arrays.append(a)  # add to feature list #
            lidar_used.append(met)  # record feature name #
            
        # --- e) Load MS indices (full site) -> resample to LiDAR grid -> then clip via plot_mask --- #
        ms_arrays = []  # store MS feature arrays #
        ms_used = []  # store MS names #
        for met, folder in MS_FOLDERS.items():  # loop MS indices #
            p = find_raster_for_site(folder, site_id, must_contain=met)  # find MS raster for site (handles inconsistent names) #
            if p is None: 
                print("Missing MS", met, "for", site_id, "-> not used")
                continue 
            a = resample_to_ref(p, ref).astype(np.float32)  # resample MS raster onto LiDAR grid #
            # IMPORTANT: the "clipping to plot buffer" is applied by only sampling pixels where plot_mask==True below #
            ms_arrays.append(a)  # add to features #
            ms_used.append(met)  # record feature name #

        # --- f) Combine all features into one (rows, cols, n_features) cube --- #
        feats = lidar_arrays + ms_arrays  # concatenate features #
        if len(feats) == 0:  # if no features loaded #
            print("No features for", site_id, "-> skipping")
            continue

        X = np.stack(feats, axis=-1)  # feature cube #
        X = np.where(np.isnan(X), -9999.0, X)  # replace NaNs with a constant sentinel #
        if not FEATURE_NAMES:  # set feature names once (first site) #
            FEATURE_NAMES = lidar_used + ms_used  # store feature names in order #

        # --- g) Define valid pixels: inside buffers + not NaN in any feature --- #
        lid_nodata = np.any(np.isnan(X[..., :len(lidar_arrays)]), axis=-1)  # because our LiDAR rasters are already clipped and valid, use LiDAR to decide “valid pixels”, not multispectral #
        valid_in_buffer = plot_mask & (~lid_nodata)  # don’t let MS NaNs wipe out pixels #

        # --- h) Extract standing/downed training pixels from labeled areas (inside buffers) --- #
        keep_sd = valid_in_buffer & (y_sd > 0)  # pixels labeled as 1 or 2 within buffers #
        X_sd = X[keep_sd]  # features for deadwood pixels #
        y_sd_vec = y_sd[keep_sd]  # labels 1/2 #

        # --- i) Sample background pixels (class 3) from unlabeled areas within buffers --- #
        bg_candidates = valid_in_buffer & (y_sd == 0)  # background candidates inside buffers, unlabeled #
        bg_idx = np.argwhere(bg_candidates)  # row/col indices of candidates #
        if len(bg_idx) == 0:  # if no background pixels available #
            print("No background candidates for", site_id, "-> continuing without bg")
            X_bg = np.empty((0, X.shape[-1]), dtype=np.float32) # empty background features #
            y_bg = np.empty((0,), dtype=np.uint8) # empty background labels #
        else:                    
        # balance idea: background count ~= deadwood count (standing+downed combined) #
            n_sd = len(y_sd_vec)  # number of labeled deadwood pixels #
            n_bg = min(len(bg_idx), max(n_sd, 1000))  # choose background size (at least some, but not more than available) #
            np.random.shuffle(bg_idx)  # randomize #
            bg_idx = bg_idx[:n_bg]  # take subset #

            X_bg = X[bg_idx[:, 0], bg_idx[:, 1]]  # background features using row/col indices #
            y_bg = np.full(len(X_bg), BG_CLASS, dtype=np.uint8)  # background labels = 3 #

        # --- j) Append site samples to global training set --- #
        if len(X_sd) == 0:  # if no deadwood pixels extracted #
            print("No standing/downed pixels in buffers for", site_id, "-> skipping")
            continue 
        X_list.append(np.vstack([X_sd, X_bg]))  # add deadwood + background #
        y_list.append(np.hstack([y_sd_vec, y_bg]))  # add labels #

## Combining all sites

In [12]:
X_all_full = np.vstack(X_list).astype(np.float32)  # merge all site samples into one feature matrix #
y_all_full = np.hstack(y_list).astype(np.uint8)  # merge all labels into one vector #

# BALANCE MULTICLASS ONLY (FOR 'Both' SCENARIO) #
# Keep SAME variable names X_all, y_all for the BALANCED multiclass set #
# This avoids the model overfitting to the majority class and improves minority-class #

X_all = X_all_full.copy()  # start from full #
y_all = y_all_full.copy()  # start from full ##

cls_ids = [1, 2, 3]  # expected classes #
cls_counts = {c: int((y_all == c).sum()) for c in cls_ids}  # count pixels per class #
print('Class counts before balancing:', cls_counts)

n_per = min(cls_counts.values())  # target equal count per class (downsample to minority) #
bal_idx = []  # balanced sample indices #
for c in cls_ids:  # for each class #
    ii = np.where(y_all == c)[0]  # indices for this class #
    np.random.shuffle(ii) 
    bal_idx.append(ii[:n_per])  # take n_per samples #
bal_idx = np.concatenate(bal_idx)  # merge #
np.random.shuffle(bal_idx)  # shuffle final mix #

X_all = X_all[bal_idx] 
y_all = y_all[bal_idx]
print('Balanced counts (for Both):', {c: int((y_all == c).sum()) for c in cls_ids}) 

# Split features into 3 model inputs #
# X_lidar / X_ms / X_fusion MUST be built for BOTH full and balanced #
# Standing/Downed-only uses FULL (X_all_full, y_all_full), Both uses BALANCED (X_all, y_all) #

lidar_feats = list(LIDAR_FOLDERS.keys())  # LiDAR feature names as used in the stack (e.g., CHM,H95,H10,GAP2) #
ms_feats = list(MS_FOLDERS.keys())  # MS feature names as used in the stack (e.g., NDVI,SAVI,GNDVI) #

lidar_idx = [i for i, f in enumerate(FEATURE_NAMES) if f in lidar_feats]  # column indices for LiDAR features #
ms_idx = [i for i, f in enumerate(FEATURE_NAMES) if f in ms_feats]  # column indices for MS features #

# FULL (unbalanced) feature matrices for binary scenarios #
X_lidar_full  = X_all_full[:, lidar_idx]
X_ms_full     = X_all_full[:, ms_idx]
X_fusion_full = X_all_full  # all features #

# BALANCED (multiclass) feature matrices for Both scenario #
X_lidar = X_all[:, lidar_idx]  # LiDAR-only matrix #
X_ms = X_all[:, ms_idx]  # MS-only matrix #
X_fusion = X_all  # Fusion matrix #

print("Features:", FEATURE_NAMES)  # show feature order once #
print("LiDAR idx:", lidar_idx, "->", [FEATURE_NAMES[i] for i in lidar_idx])  # confirm LiDAR columns #
print("MS idx   :", ms_idx, "->", [FEATURE_NAMES[i] for i in ms_idx])  # confirm MS columns #

Class counts before balancing: {1: 2505, 2: 149964, 3: 155013}
Balanced counts (for Both): {1: 2505, 2: 2505, 3: 2505}
Features: ['CHM', 'H95', 'H10', 'CGF2m', 'CC2m', 'Hmean', 'Hmed', 'Hsd', 'IQR', 'NDVI', 'SAVI', 'GNDVI', 'GCI', 'EVI', 'MSAVI', 'NDRE', 'VARI']
LiDAR idx: [0, 1, 2, 3, 4, 5, 6, 7, 8] -> ['CHM', 'H95', 'H10', 'CGF2m', 'CC2m', 'Hmean', 'Hmed', 'Hsd', 'IQR']
MS idx   : [9, 10, 11, 12, 13, 14, 15, 16] -> ['NDVI', 'SAVI', 'GNDVI', 'GCI', 'EVI', 'MSAVI', 'NDRE', 'VARI']


## Setting up binary F1/ multiclass reporting

In [13]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report 
from sklearn.metrics import f1_score  # for binary F1 reporting #

def metrics_from_cm(cm):
    """Compute OA, UA, PA from confusion matrix"""
    cm = cm.astype(float)
    diag = np.diag(cm)
    row_sum = cm.sum(axis=1)
    col_sum = cm.sum(axis=0)

    oa = diag.sum() / cm.sum() if cm.sum() > 0 else np.nan
    pa = np.divide(diag, row_sum, out=np.full_like(diag, np.nan), where=row_sum != 0)
    ua = np.divide(diag, col_sum, out=np.full_like(diag, np.nan), where=col_sum != 0)

    return oa, ua, pa
    
def make_scenario_dataset(X, y, scenario):
    """Create (X_s, y_s) for a given scenario.
    scenario = 'Standing' (1 vs background), 'Downed' (2 vs background), 'Both' (1/2/3 multiclass)
    """
    if scenario == 'Both':  # multiclass (standing/downed/background) #
        return X, y  # already balanced in cell above #
    if scenario == 'Standing':  # standing vs background only (exclude downed pixels) #
        keep = (y == 1) | (y == 3)  # drop class 2 entirely #
        y_bin = (y[keep] == 1).astype(np.uint8)  # 1=standing, 0=background #
        X_bin = X[keep]
    elif scenario == 'Downed':  # downed vs background only (exclude standing pixels) #
        keep = (y == 2) | (y == 3)  # drop class 1 entirely #
        y_bin = (y[keep] == 2).astype(np.uint8)  # 2=downed, 0=background #
        X_bin = X[keep]
    else:
        raise ValueError('Unknown scenario: ' + str(scenario))
        
    # balance binary: Standing stays balanced-to-min; Downed uses ALL downed and matches background to downed count #
    pos_idx = np.where(y_bin == 1)[0]  # positives (standing or downed) #
    neg_idx = np.where(y_bin == 0)[0]  # background #
    np.random.shuffle(pos_idx)
    np.random.shuffle(neg_idx)

    if scenario == "Downed":  # keep ALL downed pixels, match background to that count #
        n_pos = len(pos_idx)
        sel = np.concatenate([pos_idx, neg_idx[:n_pos]])
    else:  # Standing: classic balanced subset #
        n = min(len(pos_idx), len(neg_idx))
        sel = np.concatenate([pos_idx[:n], neg_idx[:n]])

    np.random.shuffle(sel)
    
    # remap binary labels back to original codes for clarity + consistent reporting #
    if scenario == "Standing":
        y_out = np.where(y_bin[sel] == 1, 1, 3).astype(np.uint8)  # 1=standing, 3=background #
    else:  # Downed
        y_out = np.where(y_bin[sel] == 1, 2, 3).astype(np.uint8)  # 2=downed,   3=background #

    print(scenario, "-> labels:", np.unique(y_out, return_counts=True))  # quick check #
    return X_bin[sel], y_out
    
def eval_holdout_scenario(title, model, Xtr, ytr, Xv, yv, scenario):
    """Fit and print metrics for either multiclass or binary scenario."""
    model.fit(Xtr, ytr)
    yv_hat = model.predict(Xv)

    print("\n==============================")
    print(title)
    if scenario == 'Both':
        # multiclass: fixed label order (1,2,3) for consistent UA/PA reporting #
        labels = [1, 2, 3]  # standing, downed, background #
        cm = confusion_matrix(yv, yv_hat, labels=labels)  # 3x3 #
        oa, ua, pa = metrics_from_cm(cm)  # compute OA/UA/PA #
        
        rep = classification_report(yv, yv_hat, digits=3, output_dict=True)
                                   
        print(f"OA (val_accuracy): {oa:.3f}")
        print(f"PA (recall)   [Standing, Downed, Background]:", np.round(pa, 3))
        print(f"UA (precision)[Standing, Downed, Background]:", np.round(ua, 3))
        print(f"F1            [Standing, Downed, Background]:",
              [round(rep.get(str(k), {}).get("f1-score", np.nan), 3) for k in labels])

        print("\nConfusion matrix (rows=ref, cols=pred):\n", cm)
        return yv_hat, {"oa": oa, "ua": ua, "pa": pa, "report": rep, "cm": cm}                           
        
    else:
         # binary: report positive-class F1 (1) + confusion matrix #
        # NOTE: positive label depends on scenario because y labels are now {1,3} or {2,3} #
        # Compute UA/PA from 2x2 cm #
        pos_label = 1 if scenario == "Standing" else 2  # positive class code #
        labels = [pos_label, 3]  # [positive, background] #
        cm = confusion_matrix(yv, yv_hat, labels=labels)  # 2x2 in this order #
        oa, ua, pa = metrics_from_cm(cm)  # OA/UA/PA for the two classes #

        f1 = f1_score(yv, yv_hat, pos_label=pos_label)  # F1 of positive code #

        print(f"OA (val_accuracy): {oa:.3f}")
        print(f"PA (recall)   [{scenario}, Background]:", np.round(pa, 3))
        print(f"UA (precision)[{scenario}, Background]:", np.round(ua, 3))
        print(f"F1 ({scenario} as positive): {f1:.3f}")

        print("\nConfusion matrix (rows=ref, cols=pred):\n", cm)

        return yv_hat, {"oa": oa, "ua": ua, "pa": pa, "f1_pos": f1, "cm": cm}
        
_ = make_scenario_dataset(X_lidar_full, y_all_full, "Downed") # quick sanity test to show FULL downed deadwood count #      

Downed -> labels: (array([2, 3], dtype=uint8), array([149964, 149964]))


##  RANDOM FOREST CLASSIFICATION 

In [14]:
from sklearn.ensemble import RandomForestClassifier 
from sklearn.model_selection import train_test_split  
from sklearn.model_selection import RandomizedSearchCV  # RF hyperparameter tuning #
from sklearn.model_selection import StratifiedKFold  # CV folds #
from sklearn.base import clone  # make fresh estimator per run #
from sklearn.metrics import make_scorer

# --- TRAINING: 3 data inputs (LiDAR / MS / Fusion) x 3 scenarios (Standing / Downed / Both) x RF (baseline + tuned) --- #
# IMPORTANT: Standing/Downed-only use FULL datasets: X_all_full with y_all_full
#  Both uses BALANCED datasets: X_all with y_all

DATASETS_BOTH = {   #datasets for multiclass (both)
    "LiDAR": X_lidar,
    "MS": X_ms,
    "Fusion": X_fusion
} 

DATASETS_FULL = {  # datasets for binary scenarios (full)
    "LiDAR": X_lidar_full,
    "MS": X_ms_full,
    "Fusion": X_fusion_full
}

SCENARIOS = ["Standing", "Downed", "Both"]  # 3 sub-models per data source #

RF_BASE = RandomForestClassifier(
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample"
)  # RF baseline #

# --- RF tuning parameter #
RF_PARAM_DIST = { 
    "n_estimators": [300, 500, 800, 1200],
    "max_depth": [None, 10, 20, 30, 40],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", None],
    "bootstrap": [True, False]
}  # RF tuned #

RESULTS = {}  # store hold-out results for quick summary #
TUNED_PARAMS = {}  # (dataset, scenario) -> best params from RandomizedSearchCV #

for ds_name in ["LiDAR", "MS", "Fusion"]: # loop data sources #
    for sc in SCENARIOS:  # loop scenarios #
        # choose full vs balanced inputs
        if sc == "Both":
            X_src = DATASETS_BOTH[ds_name]
            y_src = y_all
        else:
            X_src = DATASETS_FULL[ds_name]
            y_src = y_all_full
            
        # a) Build the scenario dataset #
        X_sc, y_sc = make_scenario_dataset(X_src, y_src, sc) 

        # b) Train/validation split for this scenario #
        idx = np.arange(len(y_sc))  # indices #
        idx_tr, idx_val, y_tr, y_val_sc = train_test_split(
            idx, y_sc, test_size=0.25, random_state=42, stratify=y_sc
        )
        X_tr, X_val = X_sc[idx_tr], X_sc[idx_val]  # split features #


        # c.1) RF BASELINE #
        title_base = f"RF | {ds_name} | {sc} | baseline"  # run label #
        rf_run = clone(RF_BASE)  # fresh unfitted model #
        yhat, rep = eval_holdout_scenario(title_base, rf_run, X_tr, y_tr, X_val, y_val_sc, sc)  # scenario-aware eval #
        RESULTS[("RF_baseline", ds_name, sc)] = (y_val_sc, yhat, rep)  # store inside RESULTS #


        # c.2) RF TUNED #
        title_tuned = f"RF | {ds_name} | {sc} | tuned"  # run label #
        # scoring depends on scenario (explicit positive label for binary cases)
        if sc == "Standing":
            scoring = make_scorer(f1_score, pos_label=1)
        elif sc == "Downed":
            scoring = make_scorer(f1_score, pos_label=2)
        else:
            scoring = "f1_macro"

        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)  # 5-fold CV #

        rf_search = RandomizedSearchCV(
            estimator=clone(RF_BASE),  # fresh RF #
            param_distributions=RF_PARAM_DIST,
            n_iter=30, 
            scoring=scoring,
            cv=cv,
            n_jobs=-1,
            pre_dispatch=2,
            verbose=1,
            random_state=42
        )  # tuning #

        rf_search.fit(X_tr, y_tr)  # fit tuning on training split #
        TUNED_PARAMS[(ds_name, sc)] = rf_search.best_params_  # SAVE params for final mapping refit #
        best_rf = rf_search.best_estimator_  # best RF #
        print(title_tuned, "best params:", rf_search.best_params_)  # show best params #

        yhat_t, rep_t = eval_holdout_scenario(title_tuned, best_rf, X_tr, y_tr, X_val, y_val_sc, sc)  # hold-out eval #
        RESULTS[("RF_tuned", ds_name, sc)] = (y_val_sc, yhat_t, rep_t)  # store inside RESULTS #

# Save RESULTS and TUNED_PARAMS so that it does not require to rerun the 10-hour loop #
import joblib  # saving/loading python objects #

joblib.dump(RESULTS, "RF_RESULTS.joblib")  # saves y_true, y_pred, and metrics #
joblib.dump(TUNED_PARAMS, "RF_TUNED_PARAMS.joblib")  # saves best parameters #
print("Saved: RF_RESULTS.joblib and RF_TUNED_PARAMS.joblib")

Standing -> labels: (array([1, 3], dtype=uint8), array([2505, 2505]))

RF | LiDAR | Standing | baseline
OA (val_accuracy): 0.903
PA (recall)   [Standing, Background]: [0.947 0.859]
UA (precision)[Standing, Background]: [0.871 0.942]
F1 (Standing as positive): 0.908

Confusion matrix (rows=ref, cols=pred):
 [[594  33]
 [ 88 538]]
Fitting 5 folds for each of 30 candidates, totalling 150 fits
RF | LiDAR | Standing | tuned best params: {'n_estimators': 800, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 'log2', 'max_depth': None, 'bootstrap': True}

RF | LiDAR | Standing | tuned
OA (val_accuracy): 0.907
PA (recall)   [Standing, Background]: [0.952 0.863]
UA (precision)[Standing, Background]: [0.874 0.947]
F1 (Standing as positive): 0.911

Confusion matrix (rows=ref, cols=pred):
 [[597  30]
 [ 86 540]]
Downed -> labels: (array([2, 3], dtype=uint8), array([149964, 149964]))

RF | LiDAR | Downed | baseline
OA (val_accuracy): 0.563
PA (recall)   [Downed, Background]: [0.31  0.

## Create summary table with OA, UA, PA, and F1

In [15]:
# LOAD SAVED RF RESULTS #
import joblib
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

RESULTS = joblib.load("RF_RESULTS.joblib")  # load saved results #

rows = []  # store table rows #

for (model, dataset, scenario), (y_true, y_pred, rep) in RESULTS.items():

    # Overall Accuracy (OA)
    oa = accuracy_score(y_true, y_pred)
    # labels
    if scenario == "Both":
        labels = [1, 2, 3]               # Standing, Downed, Background #
    elif scenario == "Standing":
        labels = [1, 3]                  # Standing, Background #
    elif scenario == "Downed":
        labels = [2, 3]                  # Downed, Background #
    else:
        raise ValueError(f"Unknown scenario: {scenario}")
                         
    # confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    # Producer's Accuracy (PA) and User's Accuracy (UA) in the same order as labels
    diag = np.diag(cm).astype(float)
    row_sum = cm.sum(axis=1).astype(float)
    col_sum = cm.sum(axis=0).astype(float)

    pa = np.divide(diag, row_sum, out=np.full_like(diag, np.nan), where=row_sum != 0)  # PA per label #
    ua = np.divide(diag, col_sum, out=np.full_like(diag, np.nan), where=col_sum != 0)  # UA per label #

    # Irrelevant classes stay empty as NaN
    PA_standing = PA_downed = PA_background = np.nan
    UA_standing = UA_downed = UA_background = np.nan

    # Fill only the classes that exist in the scenario
    for k, lab in enumerate(labels):
        if lab == 1:  # Standing
            PA_standing = pa[k]
            UA_standing = ua[k]
        elif lab == 2:  # Downed
            PA_downed = pa[k]
            UA_downed = ua[k]
        elif lab == 3:  # Background
            PA_background = pa[k]
            UA_background = ua[k]

    # classification report for F1
    F1_standing = F1_downed = F1_background = np.nan
    report = classification_report(y_true, y_pred, output_dict=True, digits=6)

    if scenario == "Standing":
        F1_standing = report.get("1", {}).get("f1-score", np.nan)
        F1_background = report.get("3", {}).get("f1-score", np.nan)

    elif scenario == "Downed":
        F1_downed = report.get("2", {}).get("f1-score", np.nan)
        F1_background = report.get("3", {}).get("f1-score", np.nan)

    elif scenario == "Both":
        F1_standing = report.get("1", {}).get("f1-score", np.nan)
        F1_downed = report.get("2", {}).get("f1-score", np.nan)
        F1_background = report.get("3", {}).get("f1-score", np.nan)

    # Build table row 
    rows.append({
        "Model": model,
        "Dataset": dataset,
        "Scenario": scenario,
        "OA": oa,
        "PA_standing": PA_standing, 
        "PA_downed": PA_downed,
        "PA_background": PA_background,

        "UA_standing": UA_standing,
        "UA_downed": UA_downed,
        "UA_background": UA_background,
        
        "F1_standing": F1_standing,
        "F1_downed": F1_downed,
        "F1_background": F1_background
    })
    
summary_df = pd.DataFrame(rows)
summary_df = summary_df.sort_values(["Dataset","Scenario","Model"])
summary_df

#export to Excel
summary_df.to_excel("Deadwood_RF_accuracy_summary.xlsx", index=False)
print("Exported: Deadwood_RF_accuracy_summary.xlsx")

Exported: Deadwood_RF_accuracy_summary.xlsx


## Create tables for model comparison and class accuracy

In [16]:
table_model = summary_df.copy()
table_class = summary_df.copy()

# keep only relevant metrics #

table_model = table_model[
    ["Model", "Dataset", "Scenario", "OA"]
]

table_class = table_class[
    [
        "Model",
        "Dataset",
        "Scenario",
        "PA_standing",
        "UA_standing",
        "F1_standing",
        "PA_downed",
        "UA_downed",
        "F1_downed"
    ]
]

table_model = table_model.sort_values(["Dataset","Scenario"])
table_class = table_class.sort_values(["Dataset","Scenario"])

table_model
table_class

table_model.to_excel("Table_Model_Comparison.xlsx", index=False)
table_class.to_excel("Table_Class_Accuracy.xlsx", index=False)

## Choosing the final models for feature importance analysis and deadwood mapping 

In [17]:
# Output: FINAL_MODELS[(ds_name, sc)] -> fitted tuned RF model #
from sklearn.base import clone  # fresh estimator #

# 1) Refit FINAL tuned RF models on FULL scenario data (balanced) #
FINAL_MODELS = {}  # (ds_name, scenario) -> fitted tuned RF #

for ds_name in ["LiDAR", "MS", "Fusion"]:  # loop LiDAR / MS / Fusion #
    for sc in SCENARIOS:  # loop Standing / Downed / Both #
        # choose full vs balanced inputs consistently with training #
        if sc == "Both":
            X_src = DATASETS_BOTH[ds_name]
            y_src = y_all
        else:
            X_src = DATASETS_FULL[ds_name]
            y_src = y_all_full

        # a) Build scenario dataset exactly like training (includes balancing) #
        X_sc, y_sc = make_scenario_dataset(X_src, y_src, sc)

        # b) Grab best params from earlier RandomizedSearchCV run #
        best_params = TUNED_PARAMS[(ds_name, sc)]  # best params for this dataset+scenario #

        # c) Refit tuned RF on FULL scenario dataset (no CV here) #
        m = clone(RF_BASE)  # fresh RF #
        m.set_params(**best_params)  # apply tuned params #
        m.fit(X_sc, y_sc)  # fit on full balanced scenario data #

        # d) Store final fitted model #
        FINAL_MODELS[(ds_name, sc)] = m 
        print("Final tuned model ready |", ds_name, "|", sc, "| best params:", best_params) 

print("\nDone. FINAL_MODELS contains", len(FINAL_MODELS), "tuned models (expected 9).") 

Standing -> labels: (array([1, 3], dtype=uint8), array([2505, 2505]))
Final tuned model ready | LiDAR | Standing | best params: {'n_estimators': 800, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 'log2', 'max_depth': None, 'bootstrap': True}
Downed -> labels: (array([2, 3], dtype=uint8), array([149964, 149964]))
Final tuned model ready | LiDAR | Downed | best params: {'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 20, 'bootstrap': False}
Final tuned model ready | LiDAR | Both | best params: {'n_estimators': 1200, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 'log2', 'max_depth': 40, 'bootstrap': True}
Standing -> labels: (array([1, 3], dtype=uint8), array([2505, 2505]))
Final tuned model ready | MS | Standing | best params: {'n_estimators': 800, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 'log2', 'max_depth': None, 'bootstrap': True}
Downed -> labels: (array([2, 3], dtype=ui

## Feature significance / importance 

In [18]:
# FEATURE SIGNIFICANCE / IMPORTANCE (RF impurity-based)
# Uses FINAL_MODELS (tuned) for each dataset×scenario (9 models)
# Writes one Excel file per dataset with scenarios as column

# helper: choose correct feature names for each dataset type #
def feature_names_for_dataset(ds_name):  # return feature names list for a dataset #
    if ds_name == "LiDAR":  # LiDAR-only #
        return [FEATURE_NAMES[i] for i in lidar_idx], lidar_idx  # names + indices #
    if ds_name == "MS":  # MS-only #
        return [FEATURE_NAMES[i] for i in ms_idx], ms_idx 
    return FEATURE_NAMES, None  # Fusion uses all features #

# loop all datasets #
for ds_name in ["LiDAR", "MS", "Fusion"]:

    names, idx = feature_names_for_dataset(ds_name)  # feature names #

    df_all = pd.DataFrame({"feature": names})  # base table #

    for sc in SCENARIOS:  # Standing / Downed / Both #

        model = FINAL_MODELS[(ds_name, sc)]  # tuned model #
        imp = model.feature_importances_ 

        df_all[sc] = imp  # add scenario column #
        df_tmp = pd.DataFrame({"feature": names, "importance": imp}).sort_values("importance", ascending=False)

    # sort by highest importance across scenarios #
    df_all["max_importance"] = df_all[SCENARIOS].max(axis=1)
    df_all = df_all.sort_values("max_importance", ascending=False).drop(columns="max_importance")

    # save Excel #
    out_path = os.path.join(OUT_DIR, f"RF_feature_importance_{ds_name}.xlsx")
    df_all.to_excel(out_path, index=False)

    print("Saved:", out_path)

Saved: D:/Kiran/Harz_Results/Modeling_results\RF_feature_importance_LiDAR.xlsx
Saved: D:/Kiran/Harz_Results/Modeling_results\RF_feature_importance_MS.xlsx
Saved: D:/Kiran/Harz_Results/Modeling_results\RF_feature_importance_Fusion.xlsx


## Creating deadwood maps (prediction rasters) within plot buffers

In [19]:
# 9 outputs per site: 3 datasets × 3 scenarios
# Uses FINAL_MODELS[(ds_name, scenario)] (tuned) and writes per-site maps inside plot buffers
# Output nodata/outside buffers = 255
# Class meanings:
#   Standing scenario: 1=standing, 3=background, 255=nodata/outside
#   Downed scenario:   2=downed,   3=background, 255=nodata/outside
#   Both scenario:     1=standing, 2=downed, 3=background, 255=nodata/outside

# 1) Create output subfolders for the 9 model maps #
for ds_name in ["LiDAR", "MS", "Fusion"]: 
    for sc in SCENARIOS:  # Standing / Downed / Both #
        out_dir = os.path.join(OUT_DIR, f"DeadwoodMap_{ds_name}_{sc}")  # folder name #
        os.makedirs(out_dir, exist_ok=True) 

# 2) Loop sites (each site has its own LiDAR reference grid) #
for site_id in SITES:  # loop all sites #

    # a) Choose reference raster grid (CHM) #
    chm_path = find_raster_for_site(LIDAR_FOLDERS["CHM"], site_id, must_contain="CHM")  # reference grid #
    if chm_path is None:  # if missing CHM #
        continue  # skip #

    with rasterio.open(chm_path) as ref:  # open reference raster #

        # b) Select plot buffers overlapping this raster #
        buff_site = buffers_for_this_raster(gdf_plot, ref)  # subset buffers #
        if len(buff_site) == 0:  # no overlap #
            continue 

        # c) Rasterize plot buffer mask (predict only inside buffers) #
        plot_mask = rasterize_mask( 
            ref, buff_site.geometry, burn_value=1, fill_value=0, dtype=np.uint8
        ).astype(bool)  # convert to boolean #

        # d) Load LiDAR metric rasters (required for all mapping modes) #
        lidar_arrays = []  # store LiDAR features #
        for met, folder in LIDAR_FOLDERS.items():  # loop required LiDAR metrics #
            p = find_raster_for_site(folder, site_id, must_contain=met)  # locate raster #
            if p is None:  # missing metric #
                lidar_arrays = []  # invalidate #
                break  # stop reading #
            with rasterio.open(p) as src:  # open metric raster #
                lidar_arrays.append(src.read(1).astype(np.float32))  # read band #
        if len(lidar_arrays) == 0:  # missing LiDAR metrics #
            print("Skipping mapping for", site_id, "(missing LiDAR metric)")
            continue 

        # e) Load MS rasters (required for MS and Fusion mapping) #
        ms_arrays = []  # store MS features #
        ms_ok = True  # flag for MS availability #
        for met, folder in MS_FOLDERS.items():  # loop MS indices #
            p = find_raster_for_site(folder, site_id, must_contain=met)  # locate MS raster #
            if p is None:  # missing MS #
                ms_ok = False  # flag missing #
                break 
            ms_arrays.append(resample_to_ref(p, ref).astype(np.float32))  # resample to LiDAR grid #

        # f) Build full feature cube in the SAME order as training FEATURE_NAMES #
        feats_full = lidar_arrays + (ms_arrays if ms_ok else [])  # combine arrays #
        Xmap_full = np.stack(feats_full, axis=-1).astype(np.float32)  # (height, width, features) #
        Xmap_full = np.where(np.isnan(Xmap_full), -9999.0, Xmap_full)  # fill NaNs #

        # g) Flatten for model prediction #
        X2_full = Xmap_full.reshape(-1, Xmap_full.shape[-1])  # (pixels, features) #
        valid2 = plot_mask.reshape(-1)  # (pixels,) mask #

        # 3) Predict rasters for all 9 models (dataset × scenario) #
        for ds_name in ["LiDAR", "MS", "Fusion"]:  
            for sc in SCENARIOS:

                # a) Pick model #
                model = FINAL_MODELS[(ds_name, sc)]  # tuned final model #

                # b) Select correct feature columns for this dataset type #
                if ds_name == "LiDAR":  # LiDAR-only #
                    X2 = X2_full[:, lidar_idx]  # select LiDAR columns #
                elif ds_name == "MS":  # MS-only #
                    if not ms_ok:  # cannot map MS without MS rasters #
                        print("Skipping", site_id, "|", ds_name, sc, "(missing MS)") 
                        continue  
                    X2 = X2_full[:, ms_idx]  # select MS columns #
                else:  # Fusion #
                    if not ms_ok:  # fusion needs MS rasters #
                        print("Skipping", site_id, "|", ds_name, sc, "(missing MS)") 
                        continue 
                    X2 = X2_full  # use all columns #

                # c) Predict inside buffers only; outside buffers = 255 #
                pred = np.full(X2.shape[0], 255, dtype=np.uint8)  # initialize #
                pred[valid2] = model.predict(X2[valid2]).astype(np.uint8)  # classify valid pixels #
                pred_img = pred.reshape(ref.height, ref.width)  # reshape to raster #

                # d) Write output GeoTIFF #
                out_meta = ref.meta.copy()  # copy georeferencing #
                out_meta.update({"count": 1, "dtype": "uint8", "nodata": 255})  # output settings #

                out_dir = os.path.join(OUT_DIR, f"DeadwoodMap_{ds_name}_{sc}")  # folder #
                out_path = os.path.join(out_dir, f"DeadwoodMap_{ds_name}_{sc}_{site_id}.tif")  # filename #

                with rasterio.open(out_path, "w", **out_meta) as dst:  # create file #
                    dst.write(pred_img, 1)  # write band #

                print("Wrote map:", out_path) 

Wrote map: D:/Kiran/Harz_Results/Modeling_results\DeadwoodMap_LiDAR_Standing\DeadwoodMap_LiDAR_Standing_CHH1.tif
Wrote map: D:/Kiran/Harz_Results/Modeling_results\DeadwoodMap_LiDAR_Downed\DeadwoodMap_LiDAR_Downed_CHH1.tif
Wrote map: D:/Kiran/Harz_Results/Modeling_results\DeadwoodMap_LiDAR_Both\DeadwoodMap_LiDAR_Both_CHH1.tif
Wrote map: D:/Kiran/Harz_Results/Modeling_results\DeadwoodMap_MS_Standing\DeadwoodMap_MS_Standing_CHH1.tif
Wrote map: D:/Kiran/Harz_Results/Modeling_results\DeadwoodMap_MS_Downed\DeadwoodMap_MS_Downed_CHH1.tif
Wrote map: D:/Kiran/Harz_Results/Modeling_results\DeadwoodMap_MS_Both\DeadwoodMap_MS_Both_CHH1.tif
Wrote map: D:/Kiran/Harz_Results/Modeling_results\DeadwoodMap_Fusion_Standing\DeadwoodMap_Fusion_Standing_CHH1.tif
Wrote map: D:/Kiran/Harz_Results/Modeling_results\DeadwoodMap_Fusion_Downed\DeadwoodMap_Fusion_Downed_CHH1.tif
Wrote map: D:/Kiran/Harz_Results/Modeling_results\DeadwoodMap_Fusion_Both\DeadwoodMap_Fusion_Both_CHH1.tif
Wrote map: D:/Kiran/Harz_Result